In [4]:
import numpy as np
import pandas as pd
import yfinance as yf

In [2]:
ticker = "NVDA"

In [5]:
# ============================================================
# 1. Download SPY data (past 1 year)
# ============================================================
data = yf.download(
    ticker,
    period="1y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

ticker_var = yf.Ticker(ticker)

if data.empty:
    raise RuntimeError("SPY data download failed")

# Use close prices as 1D numpy array
S = pd.DataFrame(data[["Close"]].to_numpy(dtype=float))
N = len(S)
print(S)


              0
0    124.795868
1    128.644821
2    129.804489
3    133.533478
4    132.763702
..          ...
246  192.509995
247  191.130005
248  185.610001
249  180.339996
250  174.190002

[251 rows x 1 columns]


In [29]:
# ============================================================
# 2. Strike price (ATM, chosen by construction)
# ============================================================

K = float(S.iloc[N-1])
print(K)

174.19000244140625


C:\Users\kstry\AppData\Local\Temp\ipykernel_4648\3810483518.py:5: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  K = float(S.iloc[N-1])


In [14]:
# ============================================================
# 3. Time domain (years)
# ============================================================
T = 30.0 / 365.0  # one month in years
t = np.linspace(0.0, T, N, dtype=float)

In [15]:
# ============================================================
# 4. Normalized price input
# ============================================================
x = S / K
# PINN input tensor: (x, t)
X_pinn = np.column_stack((x, t))

In [16]:
# ============================================================
# 6. Risk-free rate 
# ============================================================
irx = yf.download(
    "^IRX",
    period="1mo",
    interval="1d",
    auto_adjust=False,
    progress=False
)

r = 0.05  # default fallback (5%)

In [ ]:
# ============================================================
# 7. Implied Volatility (IV)
# ============================================================
window_size = 21

sigma_data = S.pct_change().rolling(window_size).std()*(252**0.5)
print(sigma_data)
sigma = float(sigma_data.iloc[250])



            0
0         NaN
1         NaN
2         NaN
3         NaN
4         NaN
..        ...
246  0.251279
247  0.252435
248  0.272253
249  0.284923
250  0.305850

[251 rows x 1 columns]
0    0.30585
Name: 250, dtype: float64


In [32]:
# ============================================================
# 7. PINN spatial domain bounds (lognormal, model-consistent)
# ============================================================
S0 = float(S.iloc[N-1])
n_std = 3.0  # 3-sigma bound

S_max = S0 * np.exp(n_std * sigma * np.sqrt(T))
S_min = S0 * np.exp(-n_std * sigma * np.sqrt(T))

x_max = S_max / K
x_min = S_min / K

C:\Users\kstry\AppData\Local\Temp\ipykernel_4648\1730622911.py:4: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S0 = float(S.iloc[N-1])


In [30]:
# ============================================================
# 8. Output summary
# ============================================================
print("========== PINN DATA SUMMARY ==========")
print(f"Strike K              : {K}")
print(f"Time domain [years]   : [0.0, {T}]")
print(f"Volatility sigma      : {sigma}")
print(f"Risk-free rate r      : {r}")
print(f"S_min, S_max          : {S_min}, {S_max}")
print(f"x_min, x_max          : {x_min}, {x_max}")
print(f"PINN input shape      : {X_pinn.shape}")
print(f"Time to expiry        : {T}")
print("=======================================")
print(type(K))

========== PINN DATA SUMMARY ==========
Strike K              : 174.19000244140625
Time domain [years]   : [0.0, 0.0822]
Volatility sigma      : 0.3058
Risk-free rate r      : 0.0500
S_min, S_max          : 133.90, 226.60
x_min, x_max          : 0    0.768701
Name: 250, dtype: float64, 0    1.300896
Name: 250, dtype: float64
PINN input shape      : (251, 2)
Time to expiry        : 0.0822
<class 'float'>
